# Duplicate & Family Management
Hashes files, groups duplicates/variants into families, freezes decisions for specific exclusion rules.


In [1]:
# Setup environment
%run 01_config.ipynb

import os
import hashlib
import pandas as pd


Creating output structure in: D:\SKIN CANCER/pipeline_output
Set basic random seeds to 42.


## Read Manifest if continuing from fresh kernal


In [2]:
manifest_path = os.path.join(OUTPUT_ROOT, "manifests", "authoritative_raw_manifest.csv")
if "df_reconciled" not in locals():
    manifest_path = os.path.join(OUTPUT_ROOT, "manifests", "authoritative_raw_manifest.csv")
    print("Loading reconciled manifest from disk...")
    df_reconciled = pd.read_csv(manifest_path)


Loading reconciled manifest from disk...


## Duplicate File Hashing & Family Detection


In [3]:
def compute_file_hash(filepath):
    if not os.path.exists(filepath): return None
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            h.update(chunk)
    return h.hexdigest()

def build_families(df_manifest):
    print("Computing file hashes...")
    df_manifest["file_hash"] = df_manifest["full_path"].apply(compute_file_hash)
    
    # Identify hashes that appear multiple times in the ENTIRE dataset
    global_hash_counts = df_manifest["file_hash"].dropna().value_counts()
    duplicated_hashes = set(global_hash_counts[global_hash_counts > 1].index)
    
    grouped = df_manifest.groupby("base_id_candidate")
    families = []
    for base_id, group in grouped:
        family_id = f"FAM_{base_id}"
        
        has_multiple_files = len(group) > 1
        folders_involved = group["folder_label"].unique()
        multiple_folders = len(folders_involved) > 1
        gt_labels_involved = group["gt_label"].unique()
        gt_mismatch = len(gt_labels_involved) > 1
        hashes_series = group["file_hash"].dropna()
        has_global_hash_collision = hashes_series.isin(duplicated_hashes).any()
        
        needs_review = (has_multiple_files or multiple_folders or gt_mismatch or has_global_hash_collision)
        
        fam_record = {
            "family_id": family_id, "base_id_candidate": base_id,
            "all_member_filenames": "|".join(group["file_name_with_extension"].astype(str)),
            "all_member_paths": "|".join(group["full_path"].astype(str)),
            "all_member_folder_labels": "|".join(group["folder_label"].astype(str)),
            "all_member_gt_labels": "|".join(group["gt_label"].astype(str)),
            "all_member_match_status": "|".join(group["match_status"].astype(str)),
            "all_member_lesion_ids": "|".join(group["lesion_id"].dropna().astype(str)),
            "all_member_hashes": "|".join(group["file_hash"].dropna().astype(str)),
            "needs_review": needs_review,
            "review_status": "pending_if_needed", "review_decision": "", "review_notes": ""
        }
        families.append(fam_record)
        df_manifest.loc[group.index, "family_id"] = family_id
        
    return df_manifest, pd.DataFrame(families)

df_manifest_w_hashes, df_families = build_families(df_reconciled)
df_families[df_families["needs_review"]==True].head(3)


Computing file hashes...


,family_id,base_id_candidate,all_member_filenames,all_member_paths,all_member_folder_labels,all_member_gt_labels,all_member_match_status,all_member_lesion_ids,all_member_hashes,needs_review,review_status,review_decision,review_notes
1247,FAM_ISIC_0012127,ISIC_0012127,ISIC_0012127_downsampled.jpg,D:\SKIN CANCER\DS\NV\ISIC_0012127_downsampled.jpg,NV,NV,matched_exact,,7675fa6a4beaf4a1cf77d9be3dfdc10ea525cd7532831d...,True,pending_if_needed,,
1484,FAM_ISIC_0012970,ISIC_0012970,ISIC_0012970_downsampled.jpg,D:\SKIN CANCER\DS\NV\ISIC_0012970_downsampled.jpg,NV,NV,matched_exact,MSK4_0011472,7675fa6a4beaf4a1cf77d9be3dfdc10ea525cd7532831d...,True,pending_if_needed,,
2567,FAM_ISIC_0024366,ISIC_0024366,ISIC_0024366.jpg,D:\SKIN CANCER\DS\NV\ISIC_0024366.jpg,NV,NV,matched_exact,HAM_0002300,b8b37d24e294c5182a1884741c109d8b3bc58edaa993e8...,True,pending_if_needed,,


## Archive Legacy Queues


In [4]:
def generate_review_queue(df_families):
    queue_path = os.path.join(OUTPUT_ROOT, "duplicate_review", "manual_adjudication_queue.csv")
    archive_path = os.path.join(OUTPUT_ROOT, "duplicate_review", "archived_manual_adjudication_queue_sections_2_3.csv")
    df_queue = df_families[df_families["needs_review"] == True].copy()
    
    # Generate legacy archive. We treat 56 excluded as permanent without pending review.
    df_queue.to_csv(archive_path, index=False)
    
    # Empty queue because Section 2/3 are CLOSED.
    pd.DataFrame(columns=df_queue.columns).to_csv(queue_path, index=False)
    print(f"Archived former queue history to {archive_path}")
    print(f"Cleared pending queue active file at {queue_path}")
    return df_queue

df_queue = generate_review_queue(df_families)


Archived former queue history to D:\SKIN CANCER/pipeline_output\duplicate_review\archived_manual_adjudication_queue_sections_2_3.csv
Cleared pending queue active file at D:\SKIN CANCER/pipeline_output\duplicate_review\manual_adjudication_queue.csv


## Apply Freezes and Generate Output Manifests


In [5]:
def apply_review_decisions(df_manifest, df_families):
    # 1. Drop old final status schema if they exist to be rerun-safe
    cols_to_drop = ["final_dataset_status", "final_exclusion_reason", "eligible_for_training", "eligible_for_split", 
                    "family_review_status", "family_review_decision", "remove_due_to_family_review_flag", "canonical_keep_flag"]
    df_manifest.drop(columns=[c for c in cols_to_drop if c in df_manifest.columns], inplace=True)

    # 2. Extract unresolved
    unresolved_fam_ids = df_families[df_families["needs_review"] == True]["family_id"].tolist()
    
    def set_exclusion_status(row):
        is_excluded = row["family_id"] in unresolved_fam_ids
        return pd.Series([
            "excluded" if is_excluded else "eligible",
            "duplicate_family_permanently_excluded" if is_excluded else "none",
            False if is_excluded else True,
            False if is_excluded else True
        ], index=["final_dataset_status", "final_exclusion_reason", "eligible_for_training", "eligible_for_split"])
        
    status_df = df_manifest.apply(set_exclusion_status, axis=1)
    df_manifest = pd.concat([df_manifest, status_df], axis=1)
    
    clean_manifest_path = os.path.join(OUTPUT_ROOT, "manifests", "cleaned_working_manifest.csv")
    df_manifest.to_csv(clean_manifest_path, index=False)
    
    training_manifest_path = os.path.join(OUTPUT_ROOT, "manifests", "training_eligible_manifest.csv")
    df_training = df_manifest[df_manifest["eligible_for_training"] == True]
    df_training.to_csv(training_manifest_path, index=False)
    
    df_excluded = df_manifest[df_manifest["eligible_for_training"] == False].copy()
    df_resolution = pd.DataFrame()
    df_resolution["full_path"] = df_excluded["full_path"]
    df_resolution["file_name_with_extension"] = df_excluded["file_name_with_extension"]
    df_resolution["normalized_id"] = df_excluded["base_id_candidate"] 
    df_resolution["class_label"] = df_excluded["gt_label"]
    df_resolution["family_id"] = df_excluded["family_id"]
    df_resolution["duplicate_hash_group"] = df_excluded["file_hash"]
    df_resolution["final_decision"] = "exclude"
    df_resolution["final_exclusion_reason"] = df_excluded["final_exclusion_reason"]
    df_resolution["notes"] = "Permanently excluded during Section 2/3 freeze"
    
    resolution_path = os.path.join(OUTPUT_ROOT, "duplicate_review", "final_duplicate_resolution.csv")
    df_resolution.to_csv(resolution_path, index=False)
    
    total_reconciled = len(df_manifest)
    total_excluded = len(df_excluded)
    final_eligible = len(df_training)
    
    print(f"Applied final freeze. Excluded {total_excluded} files.")
    print(f"Remaining pure files: {final_eligible}")
    print(f"Clean (status-rich) manifest saved to: {clean_manifest_path}")
    print(f"Training eligible manifest saved to: {training_manifest_path}")
    print(f"Final resolution saved to: {resolution_path}")
    
    return df_manifest

df_clean_manifest = apply_review_decisions(df_manifest_w_hashes, df_families)
df_clean_manifest.head(3)


Applied final freeze. Excluded 56 files.
Remaining pure files: 20664
Clean (status-rich) manifest saved to: D:\SKIN CANCER/pipeline_output\manifests\cleaned_working_manifest.csv
Training eligible manifest saved to: D:\SKIN CANCER/pipeline_output\manifests\training_eligible_manifest.csv
Final resolution saved to: D:\SKIN CANCER/pipeline_output\duplicate_review\final_duplicate_resolution.csv


,full_path,folder_name,raw_folder_label,file_name_with_extension,file_stem_raw,base_id_candidate,recognized_suffix,extension,file_exists,file_size_bytes,...,metadata_row_found,review_status,review_decision,review_notes,file_hash,family_id,final_dataset_status,final_exclusion_reason,eligible_for_training,eligible_for_split
0,D:\SKIN CANCER\DS\NV\ISIC_0000000.jpg,NV,NV,ISIC_0000000.jpg,ISIC_0000000,ISIC_0000000,NaN,.jpg,True,49964,...,True,provisionally_approved,NaN,NaN,153b10f7b8b82bf80badbfdf73a544312637a1950e2dad...,FAM_ISIC_0000000,eligible,none,True,True
1,D:\SKIN CANCER\DS\NV\ISIC_0000001.jpg,NV,NV,ISIC_0000001.jpg,ISIC_0000001,ISIC_0000001,NaN,.jpg,True,38941,...,True,provisionally_approved,NaN,NaN,6180745ca3044c6267b58dd77ae821fca7df549c64bb65...,FAM_ISIC_0000001,eligible,none,True,True
2,D:\SKIN CANCER\DS\NV\ISIC_0000003.jpg,NV,NV,ISIC_0000003.jpg,ISIC_0000003,ISIC_0000003,NaN,.jpg,True,45774,...,True,provisionally_approved,NaN,NaN,e3092bd47d1955c117a18acb1e236222cae99acfbd393d...,FAM_ISIC_0000003,eligible,none,True,True
